# GOT-OCR 2.0 — Colab API Server

1. Runtime -> Change runtime type -> **T4 GPU** sec
2. Tum hucreleri sirasiyla calistir
3. Son hucredeki URL'yi kopyala
4. Streamlit app'indeki URL kutusuna yapistir

In [ ]:
!pip install -q transformers==4.37.2 verovio fastapi uvicorn python-multipart pyngrok Pillow nest_asyncio
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import os, sys, glob
from transformers import AutoProcessor
import torch

MODEL_ID = 'stepfun-ai/GOT-OCR2_0'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float16 if DEVICE == 'cuda' else torch.float32

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

model_dirs = glob.glob(os.path.expanduser('~/.cache/huggingface/modules/transformers_modules/stepfun-ai/GOT-OCR2_0/*/'))
if not model_dirs:
    raise RuntimeError('Model kodu indirilemedi! Colab runtime restart deneyin.')
sys.path.insert(0, model_dirs[0])

from modeling_GOT import ForCausalLM

print(f'Model yukleniyor... (device={DEVICE}, dtype={DTYPE})')
model = ForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True, device_map='auto'
).to(DEVICE)
print('Model yuklendi!')

In [ ]:
from PIL import Image
import io

def run_ocr(image_bytes, max_new_tokens=4096):
    image = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    prompt = 'OCR with format: plain text'
    inputs = processor(image, prompt, return_tensors='pt').to(DEVICE, dtype=DTYPE)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, eos_token_id=processor.tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return processor.decode(generated, skip_special_tokens=True)

print('OCR fonksiyonu hazir!')

In [ ]:
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
import uvicorn, nest_asyncio

nest_asyncio.apply()
app = FastAPI(title='GOT-OCR API')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True, allow_methods=['*'], allow_headers=['*'])

@app.get('/health')
async def health():
    return {'status': 'ok', 'model': 'GOT-OCR2_0', 'device': DEVICE}

@app.post('/ocr')
async def ocr_endpoint(file: UploadFile = File(...)):
    try:
        contents = await file.read()
        if len(contents) > 10 * 1024 * 1024:
            raise HTTPException(status_code=400, detail='Dosya cok buyuk (max 10MB)')
        return {'text': run_ocr(contents), 'status': 'ok'}
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

print('FastAPI hazir!')

In [ ]:
from pyngrok import ngrok
import threading, time

public_url = ngrok.connect(8000, 'http')
print(f'\n{"="*60}')
print(f'GOT-OCR API AKTIF!')
print(f'Public URL: {public_url}')
print(f'\nBu URLyi Streamlit appindeki GOT-OCR API URL alanina yapistirin.')
print(f'{"="*60}\n')

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print('Sunucu baslatildi!')

In [ ]:
import requests
try:
    r = requests.get(f'{public_url}/health', timeout=5)
    print(f'Health: {r.json()}')
except Exception as e:
    print(f'Hata: {e}, 5sn bekleyip tekrar deneyin.')